<a href="https://colab.research.google.com/github/emilymhudson/MLOps-orchestration-pipeline/blob/main/MLOps_Streaming_Dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Asynchronous Inference Microservice

In [1]:
%%writefile async_mlops_inference_node.py
import asyncio
import json
import time
import random
import numpy as np
from datetime import datetime

class ModelServingEngine:
    """
    Simulates a production-loaded ML model.
    Implements Dynamic Batching.
    """
    def __init__(self, batch_size=32, max_latency_ms=50):
        self.batch_size = batch_size
        self.max_latency = max_latency_ms / 1000.0
        self.queue = asyncio.Queue()
        print(f"[+] Model Serving Engine initialized. (Batch Size: {self.batch_size})")

    async def _process_batch(self, batch):
        """Simulates the GPU inference and probability calibration time."""
        # Simulate network/GPU latency
        await asyncio.sleep(0.045)

        results = []
        for item in batch:
            # Simulate the calibrated output from our probabilistic classifier
            calibrated_score = round(random.uniform(0.01, 0.99), 4)
            severity = "BLOCK" if calibrated_score > 0.85 else ("WARN" if calibrated_score > 0.4 else "PASS")

            results.append({
                "log_id": item['log_id'],
                "calibrated_threat_score": calibrated_score,
                "action": severity,
                "processed_at": datetime.utcnow().isoformat()
            })
        return results

    async def inference_worker(self):
        """Runs continuously pulling from the queue and executing batch inference."""
        while True:
            batch = []
            start_time = time.time()

            # Gather logs until the batch size OR the max latency limit
            while len(batch) < self.batch_size:
                try:
                    # Wait for an item, but don't wait longer than the max latency window
                    timeout = max(0, self.max_latency - (time.time() - start_time))
                    item = await asyncio.wait_for(self.queue.get(), timeout=timeout)
                    batch.append(item)
                except asyncio.TimeoutError:
                    break # Latency window closed, process.

            if batch:
                results = await self._process_batch(batch)
                for res in results:
                    # In prod, publish to an output Kafka topic
                    if res['action'] == "BLOCK":
                        print(f"  [SOC ALERT] Log {res['log_id']} | Threat: {res['calibrated_threat_score']} | ACTION: {res['action']}")

                # Mark tasks as done
                for _ in batch:
                    self.queue.task_done()

class AsyncThreatStreamer:
    """
    Mocks an enterprise message broker.
    Streams thousands of forensic MIME telemetry JSONs into the model serving queue.
    """
    def __init__(self, serving_engine):
        self.engine = serving_engine
        self.total_logs_streamed = 0

    async def simulate_inbound_stream(self, events_per_second=500):
        """Simulates inbound email telemetry."""
        print(f"[*] Opening simulated ({events_per_second} events/sec)...")
        sleep_interval = 1.0 / events_per_second

        for i in range(1, 1001): # Stream 1,000 logs for test
            mock_log = {
                "log_id": f"EVT-99X-{i:04d}",
                "timestamp": datetime.utcnow().isoformat(),
                "payload_size_kb": random.randint(12, 105)
            }
            await self.engine.queue.put(mock_log)
            self.total_logs_streamed += 1
            await asyncio.sleep(sleep_interval)

        print("[+] Inbound stream complete.")

async def main():
    print("==================================================")
    print("      MLOps STREAMING INGESTION ARCHITECTURE      ")
    print("==================================================")

    # Initialize the engine
    serving_engine = ModelServingEngine(batch_size=64, max_latency_ms=100)
    streamer = AsyncThreatStreamer(serving_engine)

    # Start the continuous inference worker in the background
    worker_task = asyncio.create_task(serving_engine.inference_worker())

    # Start the inbound stream
    start_time = time.time()
    await streamer.simulate_inbound_stream(events_per_second=800)

    # Wait for the queue to completely empty
    await serving_engine.queue.join()
    worker_task.cancel()

    total_time = time.time() - start_time
    throughput = streamer.total_logs_streamed / total_time

    print("==================================================")
    print("               PIPELINE METRICS                   ")
    print("==================================================")
    print(f"[*] Total Logs Processed: {streamer.total_logs_streamed}")
    print(f"[*] Total Execution Time: {total_time:.2f} seconds")
    print(f"[+] Throughput Achieved:  {throughput:.2f} logs/second")
    print("==================================================")

if __name__ == "__main__":
    # Standard asyncio execution block
    asyncio.run(main())

Writing async_mlops_inference_node.py


In [2]:
!python async_mlops_inference_node.py

      MLOps STREAMING INGESTION ARCHITECTURE      
[+] Model Serving Engine initialized. (Batch Size: 64)
[*] Opening simulated (800 events/sec)...
/content/async_mlops_inference_node.py:82: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/content/async_mlops_inference_node.py:34: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "processed_at": datetime.utcnow().isoformat()
  [SOC ALERT] Log EVT-99X-0002 | Threat: 0.8539 | ACTION: BLOCK
  [SOC ALERT] Log EVT-99X-0006 | Threat: 0.9688 | ACTION: BLOCK
  [SOC ALERT] Log EVT-99X-0011 | Threat: 0.9792 | ACTION: BLOCK
  [SOC ALERT] Log EVT-99X-0015 | Threat: 0.9544 | ACTION: BLOCK
  [SOC ALE